# Basic GNN:

This is the next method used, its a basic GNN, nothing fancy like the GCRNN methods which are displayed in a different notebook.

The primarly difference is the way the data is loaded in and the goal of the model.

The goal of the model is, for some particular brain that has a graph representation where nodes are connected to their k nearest neighbors (with edge weights equal to the physical distance between them) one electrode has no ecog data that is given as a node attribute, the goal of the model is to predict what the ecog data for that held out electrode is. 

Note that we only give a small subset of the ecog data as node attributes, say 50 or 25, that way only a few numbers are having to be generated making the task easier on the model.

This restructing of the data is done in the DataLoader class, so go see its implementaion and notes for more information.

In [1]:
from DataLoader import DataLoader
from tara_preprocessing import get_just_ecog_data,get_electrode_normalized_loc,clip_time_series
from tara_preprocessing import remove_duplicates, preprocessing,apply_car_function
from noah_production_funcs_2 import get_1_patient_locations,sample_iterators,built_iterator_list
from pathlib import Path
import seaborn as sns
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
import tqdm
from torch.nn import Linear,LeakyReLU
import torch.nn.functional as F
import torch_geometric
from torch_geometric.nn import GCNConv,BatchNorm
from torch_geometric.nn import global_mean_pool,global_add_pool

### Load in the data

In [2]:
data_root = Path("/Users/noahwanless/Desktop/Spring2026/M467/faces_basic/data")
registered_dir = Path("../SuperEeg-M467-project/registered_outputs")
ecogs = get_just_ecog_data(registered_dir,data_root)
xyz = get_electrode_normalized_loc(registered_dir)
print('Downloaded data')
ecogs = clip_time_series(ecogs)
print("Time series clipped")
ecogs_no_dups,xyz_no_dups = remove_duplicates(ecogs,xyz)
print('Removed duplicate electrodes')
xyz_clean, ecogs_cleaned = preprocessing(ecogs_no_dups,xyz_no_dups,notch_size=.05)
print("Done Preprocessing")

[PosixPath('../SuperEeg-M467-project/registered_outputs/aa_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ap_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ca_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/de_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/fp_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ha_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/ja_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/jm_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/jt_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/mv_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_outputs/rn_xslocs_registered_mm.npy'), PosixPath('../SuperEeg-M467-project/registered_output

### Turns into z score

We do this manuel here, remember that in previous methods this was done when we made predictions, so in truth we are maintaining the same convention.

In [3]:
ecogs_z = []
for pat in ecogs_cleaned:
    mean = pat.mean(axis=0,keepdims=True)
    std = pat.mean(axis=0,keepdims=True)
    temp = (pat-mean)/std
    ecogs_z.append(temp)
ecogs_cleaned = ecogs_z

### Model

Note that in the model definition we have decidied a window size of 25, with 60 as the size of the hidden channels.

In [4]:
win_size = 25
class GCNv2(torch.nn.Module):
    def __init__(self, hidden_channels,num_features_per_node,output_size):
        super(GCNv2, self).__init__()
        torch.manual_seed(12345)
        self.conv1 = GCNConv(num_features_per_node, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)
        self.conv4 = GCNConv(hidden_channels, hidden_channels)
        self.batching = BatchNorm(hidden_channels)
        self.lin = Linear(hidden_channels, output_size)

    def forward(self, x, edge_index,weights,batch=None):
        x = self.conv1(x, edge_index,weights)
        x = self.batching(x)
        x = F.relu(x)
                      
        x = self.conv2(x, edge_index,weights)
        x = self.batching(x)
        x = F.relu(x)

        x = self.conv3(x, edge_index,weights)
        x = self.batching(x)
        x = F.relu(x)
        
        x = global_mean_pool(x, batch).squeeze()
        x = self.lin(x)
        
        return x
model = GCNv2(hidden_channels=60,num_features_per_node=win_size,output_size=win_size)
optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)
loss_func = torch.nn.MSELoss()
model

GCNv2(
  (conv1): GCNConv(25, 60)
  (conv2): GCNConv(60, 60)
  (conv3): GCNConv(60, 60)
  (conv4): GCNConv(60, 60)
  (batching): BatchNorm(60, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (lin): Linear(in_features=60, out_features=25, bias=True)
)

### Training

If your interested in the implementation of: built_iterator_list or of: sample_iterators, go check out the file 'noah_production_funcs_2.py'

Additionally note we have 50 epochs and a batch size of 500.

The list 'big_loss_list', is the list of loss at every evaluation, 'epoch_loss_list' is the loss at every epoch, and 'batch_loss_list' is the loss at every batch.

In [ ]:
batch_loss_list = []
epoch_loss_list = []
big_loss_list = []
epoch_loss = 0
epoch_counter = 0
batch_loss = 0
batch_counter = 0

n_epochs = 50
batch_size = 500
#for each epoch
optimizer.zero_grad()
for epoch in range(n_epochs): 
    #build the list of iterators, one for each patient
    iterators = built_iterator_list(ecogs_cleaned,xyz_clean,win_size=win_size,safety_size = 200,k = 10,max_data_points = 10000000)
    #randomly choose from one of the iterators
    for point in sample_iterators(iterators):
        #get model data
        edges = torch.tensor(point['edges'].T)
        edge_weights = torch.tensor(point['weights'],dtype=torch.float32)
        features = torch.tensor(point['features'],dtype=torch.float32)            
        y = torch.tensor(point['target'],dtype=torch.float32)
        #forward pass
        
        y_pred = model(features,edges,edge_weights)
        loss = loss_func(y_pred,y)
        loss.backward()
        #record loss

        #optimizer.step()
        #optimizer.zero_grad() #leftovers if you didnt want to do batching
        #print(loss.detach())

        batch_loss+=loss.detach()
        epoch_loss+=loss.detach()
        epoch_counter +=1

        
        if batch_counter == batch_size: #if we get enough for the batch, 
            optimizer.step() #apply grad step
            #note down loss information
            batch_loss_list.append(batch_loss/batch_size)
            big_loss_list.append(loss.detach())
            #print(f"Batch Loss:{batch_loss/batch_size}")
            batch_counter = 0  
            batch_loss = 0
            optimizer.zero_grad()
        else:
            batch_counter+=1
        
    
    epoch_loss_list.append(epoch_loss/epoch_counter)
    print(f"Epoch:{epoch} Loss:{epoch_loss/epoch_counter}")
    epoch_counter = 0
    epoch_loss = 0